# Title

In this notebook we are going to adapt a rather complicated model to better solve the problem.

In [1]:
from pathlib import Path
import sys

# Find the project root by searching upwards for the "Data" directory.
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd())
    for p in [start] + list(start.parents):
        if (p / "Data").exists():
            return p
    raise RuntimeError('Cannot find project root (missing "Data/" directory).')

ROOT = find_repo_root()
print("Project root:", ROOT)

# Optional: make the project root importable (useful even if you don't have src/ yet)
sys.path.append(str(ROOT))



Project root: c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started


In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set seeds for reproducibility across runs.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


In [4]:
# Load train/test data using ROOT so this notebook runs from anywhere.
train_path = ROOT / "Data" / "train.csv"
test_path  = ROOT / "Data" / "test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)
train_df.head()


train_df: (7613, 5)
test_df : (3263, 4)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
# Clean text in the same way as the baseline so comparisons are fair.
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)           # keep letters/numbers/spaces
    text = re.sub(r"\s+", " ", text).strip()           # normalize whitespace
    return text

train_df["text_cleaned"] = train_df["text"].apply(clean_text)
test_df["text_cleaned"]  = test_df["text"].apply(clean_text)

train_df[["text", "text_cleaned"]].head()


,text,text_cleaned
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...
3,"13,000 people receive #wildfires evacuation or...",13 000 people receive wildfires evacuation ord...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...


In [6]:
# Create a stratified split to keep the class ratio consistent.
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text_cleaned"],
    train_df["target"],
    test_size=0.2,
    random_state=42,
    stratify=train_df["target"],
)

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape)


X_train: (6090,) X_val: (1523,)
y_train: (6090,) y_val: (1523,)


In [7]:
# Check whether you are using GPU or CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [8]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

# Convert pandas objects into Hugging Face Datasets
train_hf = Dataset.from_dict({"text": list(X_train), "label": list(y_train)})
val_hf   = Dataset.from_dict({"text": list(X_val),   "label": list(y_val)})

print(train_hf, val_hf)


Dataset({
    features: ['text', 'label'],
    num_rows: 6090
}) Dataset({
    features: ['text', 'label'],
    num_rows: 1523
})


In [9]:
# Define the pre-trained model checkpoint (same as your exploration.ipynb idea)
model_ckpt = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Tokenization function
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        # Do NOT pad here; we will use a dynamic padding collator
    )

# Tokenize datasets
train_tok = train_hf.map(tokenize_batch, batched=True)
val_tok   = val_hf.map(tokenize_batch, batched=True)

# Remove raw text to avoid carrying extra columns into the Trainer
train_tok = train_tok.remove_columns(["text"])
val_tok   = val_tok.remove_columns(["text"])

# Tell HF to return PyTorch tensors
train_tok.set_format("torch")
val_tok.set_format("torch")

# Dynamic padding for each batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map: 100%|██████████| 1523/1523 [00:00<00:00, 11719.67 examples/s]


In [10]:
# Load model for sequence classification
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2).to(device)

print("Using device:", device)

# Compute metrics for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds)}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cpu


In [12]:
import sys
!{sys.executable} -m pip install -U transformers accelerate datasets



'c:\Users\Ziqian' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [15]:
output_dir = ROOT / "models" / "distilbert_run1"
output_dir.mkdir(parents=True, exist_ok=True)

# NOTE:
# Older versions of transformers may not support:
# - evaluation_strategy
# - save_strategy
# - load_best_model_at_end
# So we keep TrainingArguments minimal and do evaluation manually.
training_args = TrainingArguments(
    output_dir=str(output_dir),

    # Training hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,

    # Logging
    logging_steps=50,
    report_to="none",

    # Mixed precision if GPU is available
    fp16=torch.cuda.is_available(),
)



In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Manually evaluate after training
eval_result = trainer.evaluate()
print("Eval:", eval_result)



C:\Users\Ziqian Wang\AppData\Local\Temp\ipykernel_10736\2126720951.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.615400
100,0.455500
150,0.543000
200,0.474500
250,0.437200
300,0.425400
350,0.430100
400,0.402300
450,0.462800
500,0.413600


c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Eval: {'eval_loss': 0.43341168761253357, 'eval_f1': 0.8170144462279294, 'eval_runtime': 6.3446, 'eval_samples_per_second': 240.047, 'eval_steps_per_second': 15.131, 'epoch': 2.0}


In [17]:
from datasets import Dataset
import numpy as np
import pandas as pd

# 1. Prepare test dataset for Hugging Face Trainer
test_hf = Dataset.from_dict({
    "text": list(test_df["text_cleaned"])
})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
    )

test_tok = test_hf.map(tokenize_batch, batched=True)

# Remove raw text column and set torch format
test_tok = test_tok.remove_columns(["text"])
test_tok.set_format("torch")

# 2. Run prediction
test_pred = trainer.predict(test_tok)
logits = test_pred.predictions
test_labels = np.argmax(logits, axis=-1)

# 3. Create submission DataFrame
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "target": test_labels
})

# 4. Ensure submissions directory exists
submissions_dir = ROOT / "submissions"
submissions_dir.mkdir(parents=True, exist_ok=True)

# 5. Save submission file
output_path = submissions_dir / "submission_distilbert.csv"
submission_df.to_csv(output_path, index=False)

print(f"✅ Submission file successfully saved to: {output_path}")
submission_df.head()


Map: 100%|██████████| 3263/3263 [00:00<00:00, 67603.62 examples/s]
c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Submission file successfully saved to: c:\Users\Ziqian Wang\Desktop\William\Kaggle\nlp-getting-started\submissions\submission_distilbert.csv


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
